# Burn severity and erosion priority after the Vesuvius fire of August 2025

**What happened.** On the evening of 8 August 2025 a fire started above Terzigno, on the south-east flank of Vesuvius, and burned for four days inside the national park. The Copernicus Emergency Management Service opened activation EMSR830 for it on 9 August. The press reported about 500 hectares of pine and shrub lost.

**The question.** Which burned slopes should the park authority and the Campania civil protection treat first, before the autumn rains wash the bare ash off the steep ground?

**What this notebook does**
- Downloads Sentinel-2 images from before and after the fire and measures how badly each pixel burned.
- Combines the burn map with the slope from the Copernicus DEM.
- Ranks 250 m cells by severe burn on steep ground.

**What comes out**
- A ranked table of cells, top priority first.
- A GeoJSON of the flagged cells, ready for a GIS or a web map.
- A short plain-language summary for a non-EO reader, written by code from the numbers. No language model is used in the core; the LLM comes in bonus A.

**Data and reproducibility**
- Every pixel comes from Microsoft Planetary Computer (Sentinel-2 L2A and the Copernicus DEM GLO-30) and is processed here. Nothing is computed on a remote server.
- No account or API key is needed.

**How to run**
1. Python 3.12: `pip install -r requirements.txt`, then `pip install -e .`
2. Run all cells. The README has the Colab variant.

## 1. Where, when and the rules

Every number the analysis depends on is set once, in `src/burnsev/aoi.py`, with its reason next to it. Change one there and rerun.

**Where.** The box 14.35 to 14.50 east, 40.77 to 40.87 north: about 13 by 11 km around the cone and its south-east flank, where the fire burned.

**When**
- Pre-fire window: 1 June to 7 August 2025.
- Post-fire window: 13 August to 15 October 2025.
- Every clear scene in a window goes into a per-pixel median, so one missed cloud does not spoil the picture.

**The rules**
- Severity: dNBR in the five classes of Key and Benson (2006).
- Priority: burned at moderate or high severity (dNBR 0.27 or more in those classes) on a slope of 23 degrees or more. This is the terrain term of the USGS post-fire debris-flow model M1 (Staley et al. 2017).
- Only that terrain term is computed. The full model also needs rainfall intensity and soil erodibility.

In [1]:
# Step 1. Every number a user might want to change lives in src/burnsev/aoi.py, with its reason next to it.
from burnsev import aoi

print("AOI (west, south, east, north):", aoi.BBOX)
print("Fire:", aoi.FIRE_START, "to", aoi.FIRE_END)
print("Pre-fire window: ", *aoi.PRE_WINDOW)
print("Post-fire window:", *aoi.POST_WINDOW)
print("Bands:", aoi.BANDS, "at", aoi.RESOLUTION, "m in", aoi.CRS)
print("Severe = dNBR >=", aoi.SEVERITY_CUT_DNBR, "->", aoi.SEVERE_CLASSES)
print("Steep  = slope >=", aoi.SLOPE_THRESHOLD_DEG, "deg; sensitivity at", aoi.SLOPE_SENSITIVITY_DEG)

AOI (west, south, east, north): (14.35, 40.77, 14.5, 40.87)
Fire: 2025-08-08 to 2025-08-12
Pre-fire window:  2025-06-01 2025-08-07
Post-fire window: 2025-08-13 2025-10-15
Bands: ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12', 'SCL'] at 20 m in EPSG:32633
Severe = dNBR >= 0.27 -> ('moderate-low', 'moderate-high', 'high')
Steep  = slope >= 23.0 deg; sensitivity at (20.0, 23.0, 26.0)


## 2. Which scenes exist

**What.** A STAC search on Microsoft Planetary Computer for the Sentinel-2 L2A products that cover the box in each window. Metadata only: no pixels move yet. STAC is the standard JSON description of satellite scenes, so the same search works on any catalogue that follows it.

**Why**
- The scene table is the evidence for the whole analysis: which dates, which satellites and orbits, how cloudy, which processing version. Everything downstream can be traced back to it.
- The 25 percent cloud ceiling is a coarse filter on the whole tile. A scene can be cloudy elsewhere and clear over the box, so the real cleaning happens per pixel in step 3.
- The offset column is read from each product. Sentinel-2 changed its number format in 2022, and getting this wrong shifts every index.

In [2]:
# Step 2. Ask the catalogue which scenes exist over the box in each window. Metadata only, no pixels yet.
from burnsev import catalog

pre_items = catalog.search_scenes(aoi.BBOX, *aoi.PRE_WINDOW, aoi.MAX_CLOUD)
post_items = catalog.search_scenes(aoi.BBOX, *aoi.POST_WINDOW, aoi.MAX_CLOUD)
print(len(pre_items), "pre-fire and", len(post_items), "post-fire scenes under", aoi.MAX_CLOUD, "% cloud")

scenes = catalog.scene_table(pre_items + post_items)
scenes

32 pre-fire and 23 post-fire scenes under 25.0 % cloud


,id,date,satellite,orbit,tile,cloud_pct,baseline,boa_offset
0,S2C_MSIL2A_20250605T095051_R079_T33TVF_2025060...,2025-06-05,Sentinel-2C,79,33TVF,1.8,05.11,-1000
1,S2A_MSIL2A_20250607T095041_R079_T33TVF_2025060...,2025-06-07,Sentinel-2A,79,33TVF,0.0,05.11,-1000
2,S2A_MSIL2A_20250607T095041_R079_T33TVF_2025060...,2025-06-07,Sentinel-2A,79,33TVF,0.0,05.11,-1000
3,S2C_MSIL2A_20250608T100051_R122_T33TVF_2025060...,2025-06-08,Sentinel-2C,122,33TVF,9.5,05.11,-1000
4,S2B_MSIL2A_20250610T095029_R079_T33TVF_2025061...,2025-06-10,Sentinel-2B,79,33TVF,2.5,05.11,-1000
5,S2A_MSIL2A_20250610T100041_R122_T33TVF_2025061...,2025-06-10,Sentinel-2A,122,33TVF,2.4,05.11,-1000
6,S2B_MSIL2A_20250613T100029_R122_T33TVF_2025061...,2025-06-13,Sentinel-2B,122,33TVF,0.2,05.11,-1000
7,S2C_MSIL2A_20250615T095051_R079_T33TVF_2025061...,2025-06-15,Sentinel-2C,79,33TVF,0.6,05.11,-1000
8,S2A_MSIL2A_20250617T095051_R079_T33TVF_2025061...,2025-06-17,Sentinel-2A,79,33TVF,12.6,05.11,-1000
9,S2B_MSIL2A_20250620T095029_R079_T33TVF_2025062...,2025-06-20,Sentinel-2B,79,33TVF,1.5,05.11,-1000


**What the table shows**
- 32 products in the pre-fire window and 23 after the fire, all under 25 percent cloud over the whole tile.
- Three satellites, Sentinel-2A, 2B and the newer 2C, on two orbits, 79 and 122. The two orbits see the box from slightly different angles ten minutes apart, which is why some dates appear twice.
- First post-fire scene: 14 August, two days after containment.
- Processing baseline 05.11 for every product. The baseline is the version of the ESA software that made the product. The format changed at version 04.00 (January 2022), when 1000 was added to every stored value, so the offset is minus 1000 for all of them, read from the metadata rather than assumed.

**Two details worth knowing**
- The 7 June acquisition was published twice, 90 minutes apart. The load in step 3 groups products by solar day, so a day counts once in the median whether it has one product or two.
- The cloud filter is applied to the returned metadata in Python, because the STAC query extension is optional and this catalogue does not advertise it.

**Next.** Download the pixels of these 55 products for the seven bands, mask clouds per pixel, and convert the stored numbers to reflectance.

## 3. Load the pixels and mask the clouds

> _Narration drafted when this step is built._ odc-stac loads the cube; SCL mask; DN to reflectance with the offset; clear-pixel share per date.

In [3]:
# Step 3: written after step 2 is approved.

> _What the result shows and what it means for the next step: written after this step runs._

## 4. Look before and after

> _Narration drafted when this step is built._ True colour, pre and post, same 2 to 98 percent stretch. One figure.

In [4]:
# Step 4: written after step 3 is approved.

> _What the result shows and what it means for the next step: written after this step runs._

## 5. Composites, NBR and dNBR

> _Narration drafted when this step is built._ Median composites per window, NBR of each, dNBR = pre minus post. One figure.

In [5]:
# Step 5: written after step 4 is approved.

> _What the result shows and what it means for the next step: written after this step runs._

## 6. Severity classes and areas

> _Narration drafted when this step is built._ Key and Benson classes; map and a table of hectares per class.

In [6]:
# Step 6: written after step 5 is approved.

> _What the result shows and what it means for the next step: written after this step runs._

## 7. Check against the EFFIS perimeter

> _Narration drafted when this step is built._ Independent reference, not ground truth: area, intersection over union, where they disagree.

In [7]:
# Step 7: written after step 6 is approved.

> _What the result shows and what it means for the next step: written after this step runs._

## 8. Terrain: DEM and slope

> _Narration drafted when this step is built._ Copernicus DEM GLO-30 from the same catalogue, reprojected to the cube grid; slope in degrees.

In [8]:
# Step 8: written after step 7 is approved.

> _What the result shows and what it means for the next step: written after this step runs._

## 9. The decision: which slopes first

> _Narration drafted when this step is built._ 250 m grid, score per cell, ranked table, alert GeoJSON, plain summary; sensitivity at 20, 23 and 26 degrees.

In [9]:
# Step 9: written after step 8 is approved.

> _What the result shows and what it means for the next step: written after this step runs._

## 10. Files written

> _Narration drafted when this step is built._ COG, GeoJSON, PNGs; list with sizes; COG validated.

In [10]:
# Step 10: written after step 9 is approved.

> _What the result shows and what it means for the next step: written after this step runs._

## 11. Limitations and next steps

> _Narration drafted when this step is built._ Short list: what the numbers cannot say, what was cut and why, what comes next.

## Bonus A. The pipeline as an MCP tool, called by an LLM

> _Narration drafted when this step is built._ After the core is green.

In [11]:
# Bonus A: after the core is green.

> _What the result shows and what it means for the next step: written after this step runs._

## Bonus B. A geospatial foundation model on the same fire

> _Narration drafted when this step is built._ After bonus A.

In [12]:
# Bonus B: after bonus A.

> _What the result shows and what it means for the next step: written after this step runs._